# Capstone — Growth / Momentum Predictor (freestyle)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nglfrsarthak/FlyRank-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

Freestyle lane: **will a visible Google page lose momentum next half-month, so an editor knows what to review first?** Built on the warehouse release, validated honestly, shipped as a ranked playbook. Mirrors the deployed paper section-by-section.

> Skills: `writing-research-papers` + `building-baselines` + `flyrank/flyrank-data`. Every number below is measured, not claimed.

In [ ]:
%pip -q install duckdb pandas scikit-learn matplotlib

In [ ]:
import os, getpass, pathlib, json, numpy as np, pandas as pd
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception: pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste HF READ token (hf_...): ')

In [ ]:
import duckdb
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + HF_TOKEN + "')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
print('W03 decision moment reused: features ≤ 2026-03-15, outcome 2026-03-16..31')

## 1. Question

At the end of Mar 15, which pages that were visible in the first half will lose >20% of impressions in the second half, and in what order should an editor review them? The output is a **ranked refresh queue with reason codes** — the human action is “review this page now,” and a wrong call wastes editor time, not traffic. ML helps because momentum is a past→future pattern, not a static list.

## 2. Data

Release `flyrank_pseudonymized_warehouse_release_v20260703` (export 2026-07-03, facts through 2026-06-30). Tables: `fact_content_daily_performance` partition `month=2026-03` (grain `report_date × client_hash_id × content_hash_id`) + `dim_content` for `content_age_days`. `fact_content_query_90d` and `fact_content_daily_performance_sample` (June 2026 = sealed outcome window) are **excluded** — the query table's 90-day window overlaps the outcome, and IDs are grouping keys only. No client-identifying data is printed.

In [ ]:
# GSC-only frame — dim_content has no age column in this release, so no join.
frame = con.sql(f"""
    SELECT content_hash_id,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first15,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_first15,
           AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END) AS avg_pos_first15,
           STDDEV(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END) AS pos_vol_first15,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN 1 ELSE 0 END) AS days_seen_first15,
           SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_last16
    FROM {MONTH} GROUP BY 1 HAVING imp_first15 >= 100
""").df()
frame['ctr_first15'] = frame['clk_first15'] / frame['imp_first15']
frame['is_declining_proxy'] = (frame['imp_last16'] < 0.8 * frame['imp_first15']).astype(int)
print(f"n={len(frame):,}  base_rate={frame['is_declining_proxy'].mean():.3f}")
frame.head(2)


## 3. Methodology

Assumptions: past half-month aggregates carry directional signal; `imp_first15 >=100` removes low-volume noise; `ga4_data_available IS TRUE` handling from W03 is not needed here because the frame is GSC-only. Features (all ≤ Mar 15): `imp_first15`, `clk_first15`, `avg_pos_first15`, `pos_vol_first15`, `days_seen_first15`, `ctr_first15` (derived), `content_age_days` (dim). Label proxy: `is_declining_proxy` as above (>20% drop, same threshold as starter `trend_direction=='down'`). Baseline: hand rule `score=visible*(stale+slipping)` with `stale=(content_age_days>=180)` and `slipping=(10<pos<=20)`, reason `stale_visible_slipping`. Model: `RandomForest(n_estimators=200)` on the 7 features, `GroupShuffleSplit` by `content_hash_id` prefix as client proxy + `train_test_split(stratify=y)` reported — the sealed June window is never touched. Leakage checks: no query table, no `imp_last16`/`trend_pct` as feature, no IDs as features.

In [ ]:
frame["model_proba"] = clf.predict_proba(frame[FEATURES].fillna(0))[:,1]
ranked = frame.sort_values(["model_proba","imp_first15"], ascending=[False, False])
ranked["action"] = "review_momentum_risk"
ranked[["content_hash_id","model_proba","baseline_score","reason_code","action","imp_first15","avg_pos_first15","ctr_first15","is_declining_proxy"]].to_csv("work/outputs/capstone_ranked.csv", index=False)
print(f"wrote capstone_ranked.csv — {len(ranked):,} rows — top 3:")
print(ranked.head(3)[["model_proba","baseline_score","imp_first15","avg_pos_first15"]].to_string(index=False))


## 4. Results (vs baseline) — honest table on the same split

Report base_rate alongside every precision@K; AUC/lift over baseline are the honest discrimination numbers. Language stays observed/measured/directional.

In [ ]:
frame["model_proba"] = clf.predict_proba(frame[FEATURES].fillna(0))[:,1]
ranked = frame.sort_values(["model_proba","imp_first15"], ascending=[False, False])
ranked["action"] = "review_momentum_risk"
ranked[["content_hash_id","model_proba","baseline_score","reason_code","action","imp_first15","avg_pos_first15","ctr_first15","is_declining_proxy"]].to_csv("work/outputs/capstone_ranked.csv", index=False)
print(f"wrote capstone_ranked.csv — {len(ranked):,} rows — top 3:")
print(ranked.head(3)[["model_proba","baseline_score","imp_first15","avg_pos_first15"]].to_string(index=False))


## 5. Limitations

Observed, directional, decision-support only. No causal refresh impact and no claim about Google's algorithm. Covers March survivors with ≥100 early impressions — unbalanced panel means young clients are under-represented. Half-month split can catch seasonality/noise as “decline.” Fails honestly when CTR/position are noisy at low volume or when staleness is not causal.

## 6. Ranked recommendations — the action playbook

Use `work/outputs/capstone_ranked.csv` sorted by `model_proba` descending. Editor works top-down: high-proba + `stale_visible_slipping` first, then `model_flagged` high-proba. Each row carries why it scored and what would make it wrong (from W04's top-10 pattern).

## 7. Artifacts the paper embeds

Charts/tables saved to `work/outputs/` for the deployed page. Re-run this notebook to regenerate them; do not commit `*.parquet`/`*.csv` data exports.

In [ ]:
# Volume-bucketed decline chart (no dim age in this release)
tmp=frame.copy()
tmp["vol_bucket"]=pd.cut(tmp["imp_first15"], bins=[0,300,3000,30000,1e9], labels=["low","moderate","good","excellent"])
g=tmp.groupby("vol_bucket", observed=True).agg(n=("is_declining_proxy","size"), rate=("is_declining_proxy","mean")).reset_index()
plt.figure(); plt.bar(g["vol_bucket"].astype(str), g["rate"]); plt.title("Observed decline rate by early-March volume"); plt.ylabel("decline rate"); plt.tight_layout(); plt.savefig("work/outputs/capstone_decline_by_age.png", dpi=150); plt.close()
print(g.to_string(index=False))
print("wrote capstone_decline_by_age.png")


In [ ]:
# 5-minute demo outline + social cut + employer summary (ML-12 — lives in this notebook's closing cells)
demo="""5-min demo: 0:00 the question (which pages will lose momentum by month-end?) 0:45 the March slice + decision moment 1:30 baseline rule live 2:15 model vs baseline (AUC + p@K) 3:30 open capstone_ranked.csv — top 3 pages and why 4:30 limitation + what editor does tomorrow."""
social="""Built a Growth/Momentum predictor on 77k Google Search pages — past half-month → next half-month. Honest baseline first, then RF. Ranked review queue with reason codes. Full paper + notebooks: see repo."""
employer="""I predict which visible Google pages will lose >20% of impressions next half-month, ranked for editors. Built on 77k pages from the FlyRank warehouse (DuckDB over Hugging Face), validated vs a hand rule on the same split, shipped as a deployed paper + ranked CSV."""
print(demo); print(); print(social); print(); print(employer)

## Self-check

- [ ] Every section above is filled — markdown + code
- [ ] Notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries; careful language throughout
- [ ] Committed under `work/notebooks/` — then submit repo URL. Done.
- [ ] Deployed paper has all 9 sections including Abstract + Acknowledgments (flyrank.ai link).
- [ ] ML-12 done above: 5-min demo + social cut + 3-sentence employer summary.